In [ ]:
# invoke()：阻塞式，一次性返回完整结果。适用场景：问答、批处理任务、无需实时反馈的场景。
# ainvoke()：非阻塞式，提高系统吞吐量。适用场景：高并发Web应用、IO密集型任务。
# stream()：流式输出，实时返回每个token。适用场景：聊天机器人、长文本生成、需要提升用户体验的交互应用。
# astream()：非阻塞式，提高系统吞吐量。适用场景：高并发Web应用、IO密集型任务。
# batch()：批量处理多个输入。适用场景：高并发场景，需要同时处理大量请求。
# abatch()：非阻塞式，提高系统吞吐量。适用场景：高并发Web应用、IO密集型任务。

In [1]:
# 关于模型调用的invoke使用

## invoke的传参

### 文本输入

# 举例：

import os
from platform import system

from Demos.dde.ddeclient import conversation
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages.human import HumanMessage
from openai.types.beta import assistant

# 加载配置文件
load_dotenv(override=True)

QWEN_API_KEY=os.getenv("QWEN_API_KEY")
QWEN_BASE_URL=os.getenv("QWEN_BASE_URL")

# 获取大模型
model=init_chat_model(
    model="qwen2.5:3b",
    model_provider="openai",
    # temperatrue=0.7,    # 温度：越低输出内容越稳定；越高输出内容越随机。默认值：0.7，区间：0~2
    api_key=QWEN_API_KEY,
    base_url=QWEN_BASE_URL,
    max_tokens=150,
)

In [3]:
response=model.invoke("翻译如下汉字：你好世界")
print(response)

content='我是Qwen，是阿里巴巴集团研发的AI助手模型。我被设计用来回答各种问题、提供信息以及帮助用户完成各种任务。您可以问我关于历史、科学、文化、娱乐等任何问题，也可以和我聊天、讨论各种话题。您只需要告诉我您想知道什么或者想要讨论的主题，然后就可以开始我们的交流了。无论您有任何问题或者想聊的内容，请随时告诉我！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 32, 'total_tokens': 116, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 24, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen2.5:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-131', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a08e38-e28f-7232-9669-faa0498e41b7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 32, 'output_tokens': 84, 'total_tokens': 116, 'input_token_details': {'cache_read': 24}, 'output_token_details': {}}


In [7]:
# 字典列表

# 举例1：

from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",   # Ollama 默认地址
    timeout=180,                          # 本地模型慢，超时给足
)

messages = [
    {"role": "system", "content": "你是一个专业的数学老师"},
    {"role": "user", "content": "帮我解释一下什么是斐波那契数列"}
]

try:
    response = model.invoke(messages)
    print(response.content)
except KeyboardInterrupt:
    print("被中断了：可能是你按了停止，或推理太久")
except Exception as e:
    print("出错：", type(e).__name__, e)

斐波那契数列是一个非常有趣的数学序列，它在很多领域都有着广泛的应用，比如计算机科学、生物科学、经济周期、金融市场分析等。下面我们来详细解释斐波那契数列：

斐波那契数列是一个数列，通常定义为：F(0)=0, F(1)=1, F(n)=F(n-1)+F(n-2)。其中，F(n)表示斐波那契数列的第n个数。

这个数列最著名的特征是它的相邻两项之和等于下一项。数列的前几项如下：
1. F(0) = 0
2. F(1) = 1
3. F(2) = F(1) + F(0) = 1
4. F(3) = F(2) + F(1) = 2
5. F(4) = F(3) + F(2) = 3
6. F(5) = F(4) + F(3) = 5
7. F(6) = F(5) + F(4) = 8
8. F(7) = F(6) + F(5) = 13
...

可以看出，斐波那契数列的每个数字都是前两个数字的和，从第三个数字开始。

斐波那契数列的名字来源于意大利数学家斐波那契（Leonardo Fibonacci），他在研究兔子种群增长问题时发现了这个数列。虽然这个名字并不意味着斐波那契数列只有兔子，实际上它在自然界中也有许多出现的地方，比如植物的花瓣数量、树的分枝方式等。

斐波那契数列还有一些有趣的性质，比如它与黄金分割有关，数列中的比值越来越接近黄金比例（约等于1.618033988749895）。此外，斐波那契数列也与斐波那契螺旋有关，这是一种在自然界中常见的螺旋形态，如贝壳、花粉生长等。

总之，斐波那契数列不仅在数学中具有独特的性质和规律，而且在自然界和许多实际问题中都有广泛应用，是一条连接数学和自然界的重要纽带。


In [11]:
# 举例2：涉及多轮对话

model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
)

messages = [
    {"role": "system", "content": "你是一个专业的数学老师"},
    {"role": "user", "content": "帮我解释一下什么是斐波那契数列"},
    {"role": "assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题？"}
]

response = model.invoke(messages)
print(response)

content='您之前询问了什么是斐波那契数列。斐波那契数列是一个非常有趣的数列，在数学、自然界以及艺术等领域都有广泛的应用。接下来，我很乐意为您深入解释斐波那契数列。\n\n斐波那契数列是一个数列，其中的每一项都是前两项之和，通常以0和1开始。数列的前几项为：0, 1, 1, 2, 3, 5, 8, 13, 21, 34...，以此类推。\n\n具体来说，斐波那契数列的定义可以写作：\n- \\(F(0) = 0\\)\n- \\(F(1) = 1\\)\n- 对于所有的整数 \\(n \\geq 2\\) ，有 \\(F(n) = F(n-1) + F(n-2)\\)\n\n例如：\n- 第3项是第2项加上第1项的和：\\(F(3) = F(2) + F(1) = 1 + 1 = 2\\)\n- 第4项是第3项加上第2项的和：\\(F(4) = F(3) + F(2) = 2 + 1 = 3\\)\n\n这个数列在自然界中也经常出现，比如松果鳞片的排列、向日葵的种子分布等。希望这样解释对您有所帮助！如果有更多具体问题，欢迎随时提问。' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-09-11T04:36:44.8162654Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3147770600, 'load_duration': 2661100, 'prompt_eval_count': 46, 'prompt_eval_duration': 14412000, 'eval_count': 316, 'eval_duration': 3123351000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'} id='lc_run--01a08ec0-de3f-7f63-8184-5f29e626ac04-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens'

In [14]:
# 举例3：

model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
)

messages1 = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好我叫小明"},
]
response1 = model.invoke(messages1)
print(f"AI回复1：{response1.content}")

messages2 = [
    {"role": "system", "content": "我叫什么名字？"}
]
response2 = model.invoke(messages2)
print(f"AI回复2：{response2.content}")


AI回复1：你好，小明！很高兴遇到你。可以告诉我更多关于你的事情吗？或者你需要在哪些方面帮助到你呢？
AI回复2：您好，我是一个语言模型，没有特定的名字。我根据您使用的身份来回答您的问题。您想知道什么关于您的名字的信息呢？或者您想聊些什么？


In [16]:
# 作为对比，传递记忆

conversation = [
    {"role":"system","content":"你是一个非常友好的AI助手"},
    {"role":"user","content":"你好，我叫小明"}
]

response1 = model.invoke(conversation)
print(f"AI回复1{response1.content}")

# 添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"user","content":"我叫什么名字？"})

response2 = model.invoke(conversation)
print(f"AI回复2{response2.content}")


AI回复1你好，小明！很高兴认识你。有什么我可以帮助你的吗？或者你想聊点什么呢？比如你今天过得怎么样？或者你对未来有什么计划？别忘了，我在这里就是为了帮助你！
AI回复2你好，小明！你告诉过我你的名字是小明，所以我们现在认为你的名字是小明。如果你有任何问题或需要帮助，随时告诉我！


In [17]:
# 消息对象列表

# 举例1：

messages = [
    {"role": "system", "content": "你是一个专业的数学老师"},
    {"role": "user", "content": "帮我解释一下什么是斐波那契数列"},
    ]

response = model.invoke(messages)
print(response)

content='斐波那契数列是一个非常著名的数列，它在数学、自然界、艺术等多个领域都有广泛的应用。这个数列的定义很简单：数列的前两个数都是1，从第三个数开始，每个数等于前两个数的和。\n\n具体来说，斐波那契数列可以通过下面的递推关系来定义：\n- F(1) = 1\n- F(2) = 1\n- F(n) = F(n-1) + F(n-2) 对于 n > 2\n\n根据上述定义，斐波那契数列的前几项是这样的：\n- F(1) = 1\n- F(2) = 1\n- F(3) = F(2) + F(1) = 1 + 1 = 2\n- F(4) = F(3) + F(2) = 2 + 1 = 3\n- F(5) = F(4) + F(3) = 3 + 2 = 5\n- F(6) = F(5) + F(4) = 5 + 3 = 8\n- F(7) = F(6) + F(5) = 8 + 5 = 13\n\n以此类推，斐波那契数列可以无限扩展下去，每一个数都代表了前两个数的和。\n\n这个数列最初是由意大利数学家斐波那契在研究兔子繁殖问题时发现的。在实际应用中，斐波那契数列经常用来描述某些自然界中的现象，比如植物生长的螺旋排列方式，或者许多生物的生长模式。此外，在金融、经济学、计算机科学等领域也有广泛应用。' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-09-11T05:34:08.2017782Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3986342100, 'load_duration': 1261000, 'prompt_eval_count': 28, 'prompt_eval_duration': 26381000, 'eval_count': 369, 'eval_duration': 3953277000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'} id='lc_run--01a08ef5-65d1-7ec0-bd6e-dbff2503b138-0' tool_

In [7]:
# invoke返回值

# 举例1：

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
)

response=model.invoke([HumanMessage(content=" 2 + 3 * 2 = ? ")])
print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
To solve the expression \(2 + 3 \times 2\), you need to follow the order of operations, often remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division (from left to right), Addition and Subtraction (from left to right)).

In this expression, there are no parentheses or exponents. We need to perform multiplication before addition.

So, we first multiply \(3 \times 2\), and then add the result to 2.

\(3 \times 2 = 6\)

Now, add 2 to 6:

\(2 + 6 = 8\)

So, \(2 + 3 \times 2 = 8\).


In [8]:
# 美化输出

from rich import print as rprint

rprint(response)

AIMessage(
    content='To solve the expression \\(2 + 3 \\times 2\\), you need to follow the order of operations, often 
remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division (from left to right), 
Addition and Subtraction (from left to right)).\n\nIn this expression, there are no parentheses or exponents. We 
need to perform multiplication before addition.\n\nSo, we first multiply \\(3 \\times 2\\), and then add the result
to 2.\n\n\\(3 \\times 2 = 6\\)\n\nNow, add 2 to 6:\n\n\\(2 + 6 = 8\\)\n\nSo, \\(2 + 3 \\times 2 = 8\\).',
    additional_kwargs={},
    response_metadata={
        'model': 'qwen2.5:3b',
        'created_at': '2026-09-12T10:35:04.8529928Z',
        'done': True,
        'done_reason': 'stop',
        'total_duration': 1596765900,
        'load_duration': 2643500,
        'prompt_eval_count': 40,
        'prompt_eval_duration': 14351000,
        'eval_count': 149,
        'eval_duration': 1571614000,
        'logprobs': None,
        'model_name': 'qwen2.5:3b',
        'model_provider': 'ollama'
    },
    id='lc_run--01a0952f-50d8-7272-9892-58c90269338d-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={'input_tokens': 40, 'output_tokens': 149, 'total_tokens': 189}
)

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
)

response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['done_reason']}")
print(f"模型提供商: {metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI是通过模拟、扩展和增强人类智能的方式，使机器能够执行通常需要人类智能才能完成的复杂任务的技术。
使用的模型: qwen2.5:3b
结束原因: stop
模型提供商: ollama

输入 tokens: None
输出 tokens: None
总计 tokens: None
消息 ID: lc_run--01a0955d-b167-73f2-8674-26d3f62098c4-0
